In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'yt-dlp>=2024.11.18',
    'huggingface-hub>=0.26.0',
    'python-dotenv>=1.0.0',
    'pyyaml>=6.0',
    'requests>=2.32.0',
    'soundfile>=0.12.1',
    'numpy>=1.26.0',
    'pyloudnorm>=0.1.1',
    'pyarrow>=16.0.0',
], check=True)
subprocess.run(['apt-get', 'install', '-qq', '-y', 'ffmpeg'], check=True)

import glob as _glob
for _whl in _glob.glob('*.whl'):
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _whl], check=True)
        print(f'[install] installed {_whl}')
    except Exception as _e:
        print(f'[install] failed to install {_whl}: {_e}')

In [ ]:
import os, sys
sys.path.insert(0, '/kaggle/input/datasets/mirza176528/s2s-pipline-v2-0-2')

from shared.secrets import load_secrets
from shared.cf_client import CFClient
from shared.workflow_kernel import WorkflowKernel
from shared.repo_router import RepoRouter

import yaml
from pathlib import Path

CONFIG_DIR = Path('/kaggle/input/datasets/mirza176528/s2s-pipline-v2-0-2/config')
WORK_DIR   = Path('/kaggle/working')
WORK_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID       = os.environ.get('RUN_ID_OVERRIDE', 'run_20260507_001')
SESSION_ID   = 'cpu_collect_01'
SESSION_TYPE = 'cpu_collect'
SHARD_KEY    = 'cpu'

SECRETS = load_secrets(require_gemini=False)

with open(CONFIG_DIR / 'hf_repos.yaml') as f:
    repos_cfg = yaml.safe_load(f)

STAGE0_REPO   = repos_cfg['repos']['stage0_codec']['repo_id']
OVERFLOW_REPO = repos_cfg['repos']['overflow']['repo_id']

kernel = WorkflowKernel(
    run_id           = RUN_ID,
    session_id       = SESSION_ID,
    session_type     = SESSION_TYPE,
    shard_key        = SHARD_KEY,
    cf_worker_url    = SECRETS['CF_WORKER_URL'],
    cf_worker_secret = SECRETS['CF_WORKER_SECRET'],
    gpu_type         = None,
    vram_limit_gb    = 0.0,
    session_max_hours = 8.5,
)
kernel.start()
print(f'[session] {SESSION_ID} started \u2014 run={RUN_ID}')

In [ ]:
from huggingface_hub import HfApi, CommitOperationAdd
import shutil

WORK_DIR       = Path('/kaggle/working')
DOWNLOAD_DIR   = WORK_DIR / 'raw_downloads'
STANDARD_DIR   = WORK_DIR / 'standardized'
CLEAN_DIR      = WORK_DIR / 'clean_final'
CHECKPOINT_PATH = WORK_DIR / 'checkpoint_p1b.json'
FAILED_LOG     = WORK_DIR / 'failed_downloads.txt'


CONFIG_DIR     = Path('/kaggle/input/datasets/mirza176528/s2s-pipline-v2-0-2/config')
COOKIES_PATH   = WORK_DIR / 'cookies.txt'
COOKIE_CANDIDATES = [
    CONFIG_DIR / 'cookies.txt',
    Path('/kaggle/working/config/cookies.txt'),
    WORK_DIR / 'cookies.txt',
]

COOKIES_SRC = None
for candidate in COOKIE_CANDIDATES:
    if candidate.exists():
        COOKIES_SRC = candidate
        break

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
STANDARD_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)



if COOKIES_SRC is not None:
    shutil.copy2(str(COOKIES_SRC), str(COOKIES_PATH))
    print(f'[config] cookies refreshed from {COOKIES_SRC} -> {COOKIES_PATH}')
else:
    print('[config] WARNING: cookies file NOT FOUND in any location -- YouTube 403s are expected!')
    print('[config] Searched: ' + ', '.join(str(p) for p in COOKIE_CANDIDATES))

In [ ]:
def load_secrets():
    try:
        from kaggle_secrets import UserSecretsClient
        c = UserSecretsClient()
        secrets = {
            'HF_TOKEN_PRIMARY':   c.get_secret('HF_TOKEN_PRIMARY'),
            'HF_TOKEN_SECONDARY': c.get_secret('HF_TOKEN_SECONDARY'),
            'HF_TOKEN_TERTIARY':  c.get_secret('HF_TOKEN_TERTIARY'),
            'GEMINI_API_KEY':  c.get_secret('GEMINI_API_KEY_01') or c.get_secret('GEMINI_API_KEY'),
            
            'PROXY_URL':          c.get_secret('PROXY_URL'),
        }
        print('[secrets] loaded from Kaggle Secrets')
        return secrets
    except Exception:
        pass

    env_file = Path('.env')
    if env_file.exists():
        from dotenv import load_dotenv
        load_dotenv(env_file)
        print('[secrets] loaded from .env')

    required = ['HF_TOKEN_PRIMARY', 'HF_TOKEN_SECONDARY', 'HF_TOKEN_TERTIARY']
    missing = [k for k in required if not os.environ.get(k)]
    if missing:
        raise RuntimeError(f'Missing secrets: {missing}')
    secrets = {k: os.environ[k] for k in required}
    secrets['PROXY_URL'] = os.environ.get('PROXY_URL', '')
    return secrets

SECRETS   = load_secrets()
HF_TOKEN  = SECRETS['HF_TOKEN_PRIMARY']

PROXY_URL = (SECRETS.get('PROXY_URL') or '').strip()
if PROXY_URL:
    print(f'[proxy] configured for downloads: {PROXY_URL[:30]}...')
else:
    print('[proxy] no PROXY_URL configured \u2014 downloads may be rate-limited from Kaggle IPs')

with open(CONFIG_DIR / 'hf_repos.yaml') as f:
    repos_cfg = yaml.safe_load(f)

STAGE0_REPO = repos_cfg['repos']['stage0_codec']['repo_id']
HF_API      = HfApi(token=HF_TOKEN)
print(f'[config] stage0 repo: {STAGE0_REPO}')
print(f'[config] cookies: {"found at " + str(COOKIES_PATH) if COOKIES_PATH.exists() else "NOT FOUND \u2014 403s likely"}')

In [ ]:
import threading
import json
import requests
from datetime import datetime, timezone
import time


CHECKPOINT_PATH_P1B = WORK_DIR / 'checkpoint_p1b.json'

def load_p1b_checkpoint():

    if CHECKPOINT_PATH_P1B.exists():
        try:
            with open(CHECKPOINT_PATH_P1B) as f:
                state = json.load(f)
            print(f'[checkpoint] local \u2014 done={len(state["done"])} failed={len(state["failed"])} standardized={len(state["standardized"])}')
            return state
        except Exception:
            pass

    try:
        url = f'https://huggingface.co/datasets/{STAGE0_REPO}/resolve/main/checkpoint_p1b.json'
        r = requests.get(url, headers={'Authorization': f'Bearer {HF_TOKEN}'}, timeout=30)
        if r.status_code == 200:
            state = r.json()
            with open(CHECKPOINT_PATH_P1B, 'w') as f:
                json.dump(state, f)
            print(f'[checkpoint] HF fallback \u2014 done={len(state["done"])}')
            return state
    except Exception:
        pass

    print('[checkpoint] fresh start')
    return {
        'done': [],
        'failed': [],
        'standardized': [],
        'std_hf_uploaded': [],       # vid_ids whose standardized WAV has been uploaded to HF
        'cumulative_std_counted': [], # FIX: vid_ids already counted toward cumulative_std_bytes
        'stats': {
            'downloaded': 0,
            'standardized': 0,
            'failed_download': 0,
            'failed_standardize': 0,
            'too_short': 0,
            'too_small': 0,
            'cumulative_std_bytes': 0,  # cumulative standardized bytes across all cycles
            'std_hf_uploaded_count': 0, # count of standardized WAVs uploaded to HF
            'std_waves_committed': 0,   # number of upload waves committed for standardized audio
            'std_upload_gate_opened': False, # FIX: whether the 15 GB gate has been opened
        },
        'last_updated': None,
    }


cp_lock = threading.Lock()

def save_checkpoint(state, upload=False):
    with cp_lock:
        state['last_updated'] = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
        tmp = str(CHECKPOINT_PATH_P1B) + '.tmp'
        with open(tmp, 'w') as f:
            json.dump(state, f)
        os.replace(tmp, str(CHECKPOINT_PATH_P1B))

    if not upload:
        return

    for attempt in range(6):
        try:
            HF_API.upload_file(
                path_or_fileobj=json.dumps(state).encode(),
                path_in_repo='checkpoint_p1b.json',
                repo_id=STAGE0_REPO,
                repo_type='dataset',
                commit_message='p1b checkpoint',
            )
            return
        except Exception as e:
            wait = min(2 ** attempt, 60)
            print(f'[checkpoint] upload failed attempt {attempt+1}: {e} \u2014 retry in {wait}s')
            time.sleep(wait)


state = load_p1b_checkpoint()
done_set         = set(state['done'])
standardized_set = set(state['standardized'])
# Ensure new checkpoint fields exist in older checkpoints
if 'std_hf_uploaded' not in state:
    state['std_hf_uploaded'] = []
if 'cumulative_std_counted' not in state:
    state['cumulative_std_counted'] = []
if 'cumulative_std_bytes' not in state['stats']:
    state['stats']['cumulative_std_bytes'] = 0
if 'std_hf_uploaded_count' not in state['stats']:
    state['stats']['std_hf_uploaded_count'] = 0
if 'std_waves_committed' not in state['stats']:
    state['stats']['std_waves_committed'] = 0
if 'std_upload_gate_opened' not in state['stats']:
    state['stats']['std_upload_gate_opened'] = False
std_hf_uploaded_set = set(state['std_hf_uploaded'])
cumulative_std_counted_set = set(state['cumulative_std_counted'])
print(f'[checkpoint] cumulative_std_counted={len(cumulative_std_counted_set)} files already tracked')

In [ ]:
import requests, json, time

manifest_local = WORK_DIR / 'video_manifest.jsonl'

BATCH_SIZE = 100

if not manifest_local.exists():
    print('[manifest] downloading from HF...')
    for attempt in range(6):
        try:
            url = f'https://huggingface.co/datasets/{STAGE0_REPO}/resolve/main/video_manifest.jsonl'
            r = requests.get(url, headers={'Authorization': f'Bearer {HF_TOKEN}'}, timeout=120, stream=True)
            r.raise_for_status()
            with open(manifest_local, 'wb') as f:
                for chunk in r.iter_content(chunk_size=65536):
                    f.write(chunk)
            print(f'[manifest] downloaded to {manifest_local}')
            break
        except Exception as e:
            wait = min(2 ** attempt, 60)
            print(f'[manifest] download attempt {attempt+1} failed: {e} \u2014 retry in {wait}s')
            time.sleep(wait)

_all_videos = []
with open(manifest_local, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            _all_videos.append(json.loads(line))

_initial_pending = [v for v in _all_videos if v['video_id'] not in done_set]
print(f'[manifest] total={len(_all_videos)} already_done={len(done_set)} initial_pending={len(_initial_pending)}')


In [ ]:

STANDARDIZED_UPLOAD_THRESHOLD_GB = 15.0   # 15 GB batch trigger
STD_UPLOAD_WAVE_SIZE  = 500 * 1024 * 1024 # 500 MB per wave
STD_UPLOAD_BATCH_SIZE = 50                 # files per commit
STD_UPLOAD_MAX_RETRIES = 12
STD_UPLOAD_COMMIT_DELAY = 3.0


def get_standardized_size_on_disk_gb():
    """Calculate total size of standardized WAVs currently in STANDARD_DIR."""
    if not STANDARD_DIR.exists():
        return 0.0
    total = sum(f.stat().st_size for f in STANDARD_DIR.glob('*.wav'))
    return total / (1024 ** 3)


def _commit_std_wave(wave_files, wave_num, repo_id, token):
   
    global std_hf_uploaded_set

    existing = [p for p in wave_files if p.exists()]
    total_size_mb = sum(p.stat().st_size for p in existing) / 1024 / 1024

    print(f'  [std_wave {wave_num}] {len(existing)} files ({total_size_mb:.1f} MB) \u2192 {repo_id}')

    batches = [existing[i:i + STD_UPLOAD_BATCH_SIZE]
               for i in range(0, len(existing), STD_UPLOAD_BATCH_SIZE)]
    committed = 0

    for idx, batch in enumerate(batches):
        ops = []
        for p in batch:
            if p.exists():
                ops.append(CommitOperationAdd(
                    path_in_repo=f'audio/{p.name}',
                    path_or_fileobj=str(p),
                ))
        if not ops:
            continue

        for attempt in range(STD_UPLOAD_MAX_RETRIES):
            try:
                api = HfApi(token=token)
                api.create_commit(
                    repo_id=repo_id,
                    repo_type='dataset',
                    commit_message=f'std wave {wave_num} batch {idx+1}/{len(batches)} \u2014 {len(ops)} standardized WAVs',
                    operations=ops,
                )
                time.sleep(STD_UPLOAD_COMMIT_DELAY)
                committed += len(ops)

                with cp_lock:
                    for p in batch:
                        std_hf_uploaded_set.add(p.stem)
                        if p.stem not in state['std_hf_uploaded']:
                            state['std_hf_uploaded'].append(p.stem)
                        state['stats']['std_hf_uploaded_count'] += 1

                print(f'  [std_wave {wave_num}] batch {idx+1}/{len(batches)} OK \u2014 {len(ops)} files')
                break
            except Exception as e:
                if attempt == STD_UPLOAD_MAX_RETRIES - 1:
                    print(f'  [std_wave {wave_num}] batch {idx+1} FAILED: {e}')
                    break
                wait = min(2 ** attempt, 120)
                print(f'  [std_wave {wave_num}] attempt {attempt+1}/{STD_UPLOAD_MAX_RETRIES} '
                      f'failed: {e} \u2014 retry in {wait}s')
                time.sleep(wait)

    # Delete local files that were successfully committed
    for p in existing:
        if p.stem in std_hf_uploaded_set:
            p.unlink(missing_ok=True)

    with cp_lock:
        state['stats']['std_waves_committed'] += 1

    print(f'  [std_wave {wave_num}] done \u2014 {committed}/{len(existing)} committed')
    return committed


def upload_standardized_to_hf(force=False):
    """Upload standardized WAVs from STANDARD_DIR to HF audio/ subdir.

    The upload is gated by STANDARDIZED_UPLOAD_THRESHOLD_GB: it only fires
    when the cumulative standardized bytes (tracked in checkpoint) reach the
    threshold, OR when force=True (used for the final session flush).

    Once the gate has been opened (i.e., the threshold was reached in a
    previous cycle), all subsequent cycles will upload regardless of size.

    After a successful upload, the local WAV files are deleted to free disk
    space. The checkpoint is updated and pushed to HF.

    Returns the total number of files uploaded.
    """
    global std_hf_uploaded_set

    std_files = sorted(STANDARD_DIR.glob('*.wav'))
    # Filter to files not yet uploaded to HF
    pending = [f for f in std_files if f.stem not in std_hf_uploaded_set]

    if not pending:
        print('[std_upload] no pending standardized files to upload')
        return 0

    pending_size_bytes = sum(f.stat().st_size for f in pending)
    pending_size_gb = pending_size_bytes / (1024 ** 3)
    cumulative_gb = state['stats']['cumulative_std_bytes'] / (1024 ** 3)

    # Check if the upload gate has been opened (threshold reached in a prior cycle)
    gate_opened = state['stats'].get('std_upload_gate_opened', False)

    if not force and not gate_opened:
        if cumulative_gb < STANDARDIZED_UPLOAD_THRESHOLD_GB:
            print(f'[std_upload] cumulative {cumulative_gb:.2f} GB / '
                  f'{STANDARDIZED_UPLOAD_THRESHOLD_GB} GB threshold \u2014 '
                  f'waiting ({len(pending)} files, {pending_size_gb:.2f} GB on disk)')
            return 0
        else:
            # Threshold reached for the first time \u2014 open the gate
            print(f'[std_upload] threshold reached! cumulative {cumulative_gb:.2f} GB >= '
                  f'{STANDARDIZED_UPLOAD_THRESHOLD_GB} GB \u2014 opening upload gate')
            with cp_lock:
                state['stats']['std_upload_gate_opened'] = True

    print(f'[std_upload] uploading {len(pending)} standardized WAVs ({pending_size_gb:.2f} GB) to HF')

    # Determine target repo (respect overflow routing)
    current_repo  = repo_router.target_repo
    current_token = HF_TOKEN
    if repo_router.using_overflow:
        current_token = SECRETS.get('HF_TOKEN_TERTIARY', HF_TOKEN)

    # Wave-based upload
    wave_buf   = []
    wave_bytes = 0
    wave_num   = state['stats']['std_waves_committed'] + 1
    total_uploaded = 0

    for wav_path in pending:
        file_size = wav_path.stat().st_size
        wave_buf.append(wav_path)
        wave_bytes += file_size

        if wave_bytes >= STD_UPLOAD_WAVE_SIZE:
            committed = _commit_std_wave(wave_buf, wave_num, current_repo, current_token)
            total_uploaded += committed
            repo_router.record_commit(wave_bytes)
            wave_buf   = []
            wave_bytes = 0
            wave_num  += 1

    # Flush remaining files in the buffer
    if wave_buf:
        committed = _commit_std_wave(wave_buf, wave_num, current_repo, current_token)
        total_uploaded += committed
        repo_router.record_commit(wave_bytes)

    save_checkpoint(state, upload=True)
    print(f'[std_upload] done \u2014 {total_uploaded} files uploaded to HF '
          f'(total HF-uploaded: {len(std_hf_uploaded_set)})')

    return total_uploaded


print(f'[std_upload] configured: threshold={STANDARDIZED_UPLOAD_THRESHOLD_GB} GB, '
      f'wave_size={STD_UPLOAD_WAVE_SIZE/1024/1024:.0f} MB, batch={STD_UPLOAD_BATCH_SIZE}')
print(f'[std_upload] current state: cumulative={state["stats"]["cumulative_std_bytes"]/1024**3:.2f} GB, '
      f'already_uploaded={len(std_hf_uploaded_set)}, '
      f'gate_opened={state["stats"].get("std_upload_gate_opened", False)}, '
      f'cumulative_counted={len(cumulative_std_counted_set)}')

In [ ]:
import json as _json
import shutil

# --- Fix 3: Define CookieExpiredError so we can catch it from p1b ---
class CookieExpiredError(Exception):
    """Raised when 20+ consecutive cookie-rejected downloads indicate cookies are fully expired."""


_SESSION_CRITICAL_GLOBALS = (
    'load_p1b_checkpoint', 'save_checkpoint', 'cp_lock',
    'CHECKPOINT_PATH_P1B', 'load_checkpoint', 'CHECKPOINT_PATH',
    'std_hf_uploaded_set', 'upload_standardized_to_hf',
    'cumulative_std_counted_set', 'update_cumulative_std_bytes',
)

def exec_notebook(path):
    
    _saved = {k: globals()[k] for k in _SESSION_CRITICAL_GLOBALS if k in globals()}

    with open(path) as _f:
        _nb = _json.load(_f)
    for _cell in _nb.get('cells', []):
        if _cell.get('cell_type') == 'code':
            _src = ''.join(_cell.get('source', []))
            exec(_src, globals())

    
    globals().update(_saved)


def reload_checkpoint_state():
    cp = load_p1b_checkpoint()
    return set(cp['done']), set(cp['standardized']), cp['stats']


def purge_media_dirs(purge_unuploaded_std=False):
    """Purge local media directories to free disk space.

    - DOWNLOAD_DIR is always fully purged (raw downloads not needed after p1b).
    - STANDARD_DIR: only files that have been confirmed uploaded to HF are
      deleted by default. Set purge_unuploaded_std=True to force-delete
      everything (used during cookie-expired cooldown to prevent stale data).
    """
    # Always purge raw downloads
    if DOWNLOAD_DIR.exists():
        shutil.rmtree(str(DOWNLOAD_DIR))
        DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
        print(f'[cleanup] purged {DOWNLOAD_DIR}')

    # Selectively purge standardized files
    if STANDARD_DIR.exists():
        if purge_unuploaded_std:
            # Force purge everything (e.g., during cookie cooldown)
            shutil.rmtree(str(STANDARD_DIR))
            STANDARD_DIR.mkdir(parents=True, exist_ok=True)
            print(f'[cleanup] force-purged {STANDARD_DIR}')
        else:
            # Only delete files that have been confirmed uploaded to HF
            std_files = list(STANDARD_DIR.glob('*.wav'))
            deleted = 0
            kept = 0
            for f in std_files:
                if f.stem in std_hf_uploaded_set:
                    f.unlink(missing_ok=True)
                    deleted += 1
                else:
                    kept += 1
            print(f'[cleanup] {STANDARD_DIR}: deleted {deleted} uploaded, kept {kept} pending-upload')


def update_cumulative_std_bytes():
    """Add NEW standardized file sizes to the cumulative byte counter.

    FIX: Only counts files whose vid_id is NOT already in
    cumulative_std_counted_set. This prevents double-counting files that
    persist on disk across multiple cycles (because they haven't been
    uploaded to HF yet and thus weren't deleted).

    After counting, each new vid_id is added to cumulative_std_counted
    in the checkpoint so it won't be re-counted on the next cycle.
    """
    global cumulative_std_counted_set

    if not STANDARD_DIR.exists():
        return

    new_bytes = 0
    new_ids = []
    for f in STANDARD_DIR.glob('*.wav'):
        vid_id = f.stem
        if vid_id not in cumulative_std_counted_set:
            new_bytes += f.stat().st_size
            new_ids.append(vid_id)

    if new_bytes > 0:
        with cp_lock:
            state['stats']['cumulative_std_bytes'] += new_bytes
            for vid_id in new_ids:
                cumulative_std_counted_set.add(vid_id)
                if vid_id not in state['cumulative_std_counted']:
                    state['cumulative_std_counted'].append(vid_id)

    cumulative_gb = state['stats']['cumulative_std_bytes'] / (1024 ** 3)
    print(f'[std_upload] cumulative standardized: {cumulative_gb:.2f} GB '
          f'(+{new_bytes/1024**3:.2f} GB new this cycle, '
          f'{len(new_ids)} new files, '
          f'threshold: {STANDARDIZED_UPLOAD_THRESHOLD_GB} GB)')


def count_pending():
    all_vids = []
    with open(manifest_local, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                all_vids.append(json.loads(line))
    current_done = set(load_p1b_checkpoint()['done'])
    return [v for v in all_vids if v['video_id'] not in current_done]


def count_clean_pending():
    """Count files in CLEAN_DIR not yet in p1e's uploaded_set."""
    CLEAN_DIR.mkdir(parents=True, exist_ok=True)
    clean_files = sorted(CLEAN_DIR.glob('*.wav'))

    # Load p1e checkpoint to get uploaded_set
    p1e_uploaded = set()
    p1e_cp_path = WORK_DIR / 'checkpoint_p1e.json'
    if p1e_cp_path.exists():
        try:
            with open(p1e_cp_path) as f:
                p1e_state = json.load(f)
            p1e_uploaded = set(p1e_state.get('uploaded_ids', []))
        except Exception:
            pass

    return [p for p in clean_files if p.stem not in p1e_uploaded]


PIPELINE_DIR = '/kaggle/input/datasets/mirza176528/s2s-pipline-v2-0-2/pipeline_1_collect'

repo_router = RepoRouter(STAGE0_REPO, OVERFLOW_REPO)

# CHANGE 3: Batch trigger threshold for p1e uploads
P1E_BATCH_TRIGGER = 50

cycle = 0

while True:
    if kernel.session_expiring:
        print(f'[session] expiring before cycle {cycle + 1} \u2014 stopping')
        break

    remaining = count_pending()
    if not remaining:
        print(f'[session] all videos processed after {cycle} cycle(s) \u2014 done')
        break

    cycle += 1
    print(f'\n[session] === cycle {cycle} | pending={len(remaining)} ===')

    kernel.check_session_time()
    started_at = kernel.log_stage_start('p1a')
    try:
        exec_notebook(f'{PIPELINE_DIR}/p1a_discover.ipynb')
        kernel.log_stage_end('p1a', started_at)
        print(f'[session] p1a completed (cycle {cycle})')
    except Exception as e:
        kernel.log_stage_end('p1a', started_at, error=str(e))
        print(f'[session] p1a failed (cycle {cycle}): {e}')
        raise

    if kernel.session_expiring:
        print(f'[session] expiring after p1a cycle {cycle} \u2014 stopping')
        break

    remaining = count_pending()
    if not remaining:
        print(f'[session] no pending videos after p1a cycle {cycle} \u2014 done')
        break

    kernel.check_session_time()
    started_at = kernel.log_stage_start('p1b')
    try:
        exec_notebook(f'{PIPELINE_DIR}/p1b_download.ipynb')
        kernel.log_stage_end('p1b', started_at)
        print(f'[session] p1b completed (cycle {cycle})')
    except CookieExpiredError:
        kernel.log_stage_end('p1b', started_at, error='cookie_expired')
        print('[session] p1b stopped early \u2014 cookies expired. Skipping p1e for this cycle.')
        print('[session] cooling down 5 minutes before next cycle (YouTube cookie cooldown)...')
        done_set, standardized_set, _ = reload_checkpoint_state()
        # Force-purge during cookie cooldown to prevent stale data accumulation
        purge_media_dirs(purge_unuploaded_std=True)
        time.sleep(300)  # 5-minute cooldown before next cycle
        continue
    except Exception as e:
        kernel.log_stage_end('p1b', started_at, error=str(e))
        print(f'[session] p1b failed (cycle {cycle}): {e}')
        raise

    done_set, standardized_set, _ = reload_checkpoint_state()

    
    update_cumulative_std_bytes()

    
    std_uploaded = upload_standardized_to_hf(force=False)
    if std_uploaded > 0:
        print(f'[session] uploaded {std_uploaded} standardized WAVs to HF (cycle {cycle})')

    if kernel.session_expiring:
        print(f'[session] expiring after p1b cycle {cycle} \u2014 stopping before p1e')
        break

    # CHANGE 3: Counter-based p1e trigger
    # Only run p1e when >= P1E_BATCH_TRIGGER files are ready for upload
    clean_pending = count_clean_pending()
    print(f'[session] clean_final pending upload: {len(clean_pending)} files (threshold: {P1E_BATCH_TRIGGER})')

    if len(clean_pending) >= P1E_BATCH_TRIGGER:
        kernel.check_session_time()
        started_at = kernel.log_stage_start('p1e')
        # Normal run: not forced, respects BATCH_TRIGGER_SIZE inside p1e
        FORCE_UPLOAD = False
        try:
            exec_notebook(f'{PIPELINE_DIR}/p1e_upload.ipynb')
            kernel.log_stage_end('p1e', started_at)
            print(f'[session] p1e completed (cycle {cycle})')
        except Exception as e:
            kernel.log_stage_end('p1e', started_at, error=str(e))
            print(f'[session] p1e failed (cycle {cycle}): {e}')
            raise
    else:
        print(f'[session] skipping p1e \u2014 only {len(clean_pending)} files ready (need {P1E_BATCH_TRIGGER})')

    # Purge: raw downloads always; standardized only if uploaded to HF
    purge_media_dirs(purge_unuploaded_std=False)

    done_set, standardized_set, _ = reload_checkpoint_state()

    remaining_after = count_pending()
    print(f'[session] cycle {cycle} complete | remaining={len(remaining_after)}')

    if not remaining_after:
        print(f'[session] manifest fully processed in {cycle} cycle(s)')
        break

# --- Final flush: upload any remaining standardized WAVs ---
print(f'\n[session] final standardized audio flush...')
update_cumulative_std_bytes()
std_uploaded = upload_standardized_to_hf(force=True)
if std_uploaded > 0:
    print(f'[session] final std upload: {std_uploaded} files')
else:
    print('[session] no remaining standardized files to upload')

# CHANGE 3: Final flush \u2014 always run p1e with FORCE_UPLOAD=True at the end
# to upload any remaining files regardless of batch count
clean_pending = count_clean_pending()
if clean_pending:
    print(f'\n[session] final flush: {len(clean_pending)} files remaining \u2014 running p1e with FORCE_UPLOAD=True')
    FORCE_UPLOAD = True
    try:
        exec_notebook(f'{PIPELINE_DIR}/p1e_upload.ipynb')
        print('[session] final p1e flush completed')
    except Exception as e:
        print(f'[session] final p1e flush failed: {e}')
else:
    print('\n[session] no remaining files to upload \u2014 final flush not needed')

kernel.stop()
print('[session] cpu_collect session complete')
